In [1]:
#Programma 2 progetto di Linguistica Computazionale di Lavinia Rotellini, matricola: 581553, a.a. 2023-2024 
#Importo e scarico i moduli che serviranno
import nltk
import math

nltk.download('punkt')
nltk.download('maxent_ne_chunker')
nltk.download('words')

from nltk import FreqDist, ngrams
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

#Assegno il lemmatizzatore ad una variabile per snellire il codice
lemmatizer = WordNetLemmatizer() 

#Il corpus che analizzerò
file = "mrsdalloway.txt" #il corpus che analizzeremo


#LETTURA FILE
def read_file(file_path):
  with open (file_path, "r", encoding= "utf-8") as infile:
    contents = infile.read()
  return contents 


#RACCOLTA DELLE FRASI, DEI TOKEN, POS TAGGING E LEMMATIZZAZIONE DEI TESTI

#Nella funzione sotto divido il testo in frasi e token, poi realizzo il POS tagging
def analizing(text):
  sentences = nltk.tokenize.sent_tokenize(text)
  tot_tokens = []
  tot_POS = []
  for sentence in sentences:
    tokens = nltk.tokenize.word_tokenize(sentence)
    tot_tokens.extend(tokens)
    POS = nltk.tag.pos_tag(tokens)
    tot_POS.extend(POS)
  return sentences, tot_tokens, tot_POS

#Questa seconda funzione serve a trasformare la treebank delle parts of speech in una treebank leggibile secondo WordNet
def get_wordnet_pos(treebank_tag):
  if treebank_tag.startswith('J'):
    return wordnet.ADJ

  if treebank_tag.startswith('V'):
    return wordnet.VERB

  if treebank_tag.startswith('N'):
    return wordnet.NOUN

  if treebank_tag.startswith('R'):
    return wordnet.ADV

  else:
    return wordnet.NOUN

#Con la prossima funzione eseguo la funzione di analisi e individuo i lemmi
def main(file_path):
  text = read_file(file_path)
  sentences, tot_tokens, tot_POS= analizing(text)
  lemmatizzazione = [lemmatizer.lemmatize(word, get_wordnet_pos(POS)) for word, POS in tot_POS]
  return sentences, tot_tokens, tot_POS, lemmatizzazione

file_sentences, file_tokens, file_POS, file_lemma = main(file)

#LEMMATIZZAZIONE, così da avere una lista di tuple in cui abbiamo il token affiancato al lemma
def lemmatizzazione(tokens, lemmi):
  lemmatizzazione = list(zip(tokens, lemmi))
  print(f"La lista token/lemmi del testo: {lemmatizzazione}")

lemmatizzazione(file_tokens, file_lemma)

###PUNTO UNO###
##CINQUANTA SOSTANTIVI, AVVERBI ED AGGETTIVI PIU' FREQUENTI

#La funzione sotto trova tutte le istanze di sostantivo, avverbio e aggettivo
def find_POS(list_of_POS, list_of_elements):
  lista_specifica_POS = []
  for token, POS in list_of_elements: #scorriamo la lista delle parts of speech
    if POS in list_of_POS:            #se il pos nella lista è dentro l'altro parametro list of POS, inseriamo nella lista la tupla token, POS
      lista_specifica_POS.append((token, POS))
  return lista_specifica_POS

tag_sostantivi = ["NN", "NNS", "NNP", "NNPS"]         #usiamo queste liste per controllare e prendere i POS con tag dei sostantivi
lista_sostantivi = find_POS(tag_sostantivi, file_POS) #chiamiamo la funzione che ritorna la lista di sostantivi
freq_sostantivi = FreqDist(lista_sostantivi)          #calcoliamo la frequenza assoluta, il processo si ripete per aggettivi e avverbi

tag_aggettivi = ["JJ", "JJR", "JJS"]
lista_aggettivi = find_POS(tag_aggettivi, file_POS)
freq_aggettivi = FreqDist(lista_aggettivi)

tag_avverbi = [ "RB", "RBR", "RBS", "WRB"]
lista_avverbi = find_POS(tag_avverbi, file_POS)
freq_avverbi = FreqDist(lista_avverbi)

#La funzione sotto trova i cinquanta POS più comuni e li stampa
def lista_POS(lista_POS, top_n = 50):                
   for POS, freq in lista_POS.most_common(top_n):
      print(f"Part of speech: {POS} Frequenza assoluta: {freq}")

#Chiamo la funzione precedente sulle liste di sostantivi, aggettivi e avverbi
lista_POS(freq_sostantivi, top_n = 50)              
lista_POS(freq_aggettivi, top_n = 50)
lista_POS(freq_avverbi, top_n = 50)


###PUNTO DUE###
##VENTI NGRAMMI PIU' FREQUENTI

#Inizio con la divisione del testo in ngrammi
ngrams_1 = list(nltk.ngrams(file_tokens, 1))
ngrams_2 = list(nltk.ngrams(file_tokens, 2))
ngrams_3 = list(nltk.ngrams(file_tokens, 3))
ngrams_4 = list(nltk.ngrams(file_tokens, 4))
ngrams_5 = list(nltk.ngrams(file_tokens, 5)) 

#La funzione sotto individua e stampa i venti ngrammi più frequenti
def top_20_ngrams(ngrams, top_n = 20):
  freq_ngrammi = FreqDist(ngrams)
  for ngram, freq in freq_ngrammi.most_common(top_n):
    print(f"Ngramma: {ngram} \n\tFrequenza assoluta: {freq}\n")

top_20_ngrams(ngrams_1, top_n = 20)
top_20_ngrams(ngrams_2, top_n = 20)
top_20_ngrams(ngrams_3, top_n = 20)
top_20_ngrams(ngrams_4, top_n = 20)
top_20_ngrams(ngrams_5, top_n = 20)


###PUNTO TRE###
##VENTI NGRAMMI DI POS PIU' FREQUENTI

#Con il ciclo sotto prendiamo le POS senza token e le inseriamo in una lista
just_POS = []
for token, POS in file_POS: #prendiamo solo le parts of speech senza token e le inseriamo in una lista
   just_POS.append(POS)

#Dividiamo la lista di POS in unigrammi, bigrammi, trigrammi
ngrams_POS_1 = list(nltk.ngrams(just_POS, 1))
ngrams_POS_2 = list(nltk.ngrams(just_POS, 2))
ngrams_POS_3 = list(nltk.ngrams(just_POS, 3))

#Con la funzione sotto troviamo i primi venti ngrammi di POS per frequenza
def top_20_ngrams_POS(ngrams_POS, top_n = 20):
  freq_ngrammi_POS = FreqDist(ngrams_POS)
  for ngram_POS, freq in freq_ngrammi_POS.most_common(top_n):
    print(f"Ngramma: {ngram_POS} \n\tFrequenza assoluta: {freq}\n")

top_20_ngrams_POS(ngrams_POS_1, top_n= 20)
top_20_ngrams_POS(ngrams_POS_2, top_n= 20)
top_20_ngrams_POS(ngrams_POS_3, top_n= 20)


###PUNTO QUATTRO###
##DIECI BIGRAMMI AGGETTIVO SOSTANTIVO PIU' FREQUENTI

#Con il ciclo sotto estraimo le coppie di token aggettivo/sostantivo e le inseriamo in una lista
adj_noun_tuples = []                                                   
for i in range(len(file_POS)-1):
  if file_POS[i][1] in tag_aggettivi and file_POS[i+1][1] in tag_sostantivi:
    adj_noun_tuples.append((file_POS[i][0], file_POS[i+1][0]))
       
#La funzione sotto restituisce i primi dieci bigrammi in ordine di frequenza massima
def adj_noun_couples(list_of_bigrams, top_n = 10):                     
   freq_bigrammi_ADNO = FreqDist(list_of_bigrams)
   for bigram, freq in freq_bigrammi_ADNO.most_common(top_n):
      print(f"Bigrammi: {bigram} \n\tFrequenza assoluta in ordine decrescente: {freq}")

adj_noun_couples(adj_noun_tuples, top_n = 10)

###I primi dieci bigrammi per probabilità condizionata, congiunta, mutual information e local mutual information###

#La funzione sotto prende una lista, mette gli elementi in ordine decrescente e ne prende i primi dieci
def top_10(list_of_elements, funzione):
  sort_elements = sorted(list_of_elements,key=lambda item: item[1], reverse=True)
  print(f"Top 10 bigrammi per {funzione}: \n\t {sort_elements[:10]}\n")
  return sort_elements[:10]

#La funzione sotto prende in input un'altra funzione ed una lista di elementi, oltre al numero totale di bigrammi e token,
#e ad ogni elemento della lista applica la funzione. Restituisce una lista, a cui sarà applicata la funzione top 10 sopra
def liste(lista, funzione, tot_bigrams, tot_tokens):
  totale_elementi = []
  for element in lista:
    risultato = funzione(element, tot_bigrams, tot_tokens)
    totale_elementi.append((element, risultato))
  return totale_elementi

#Creo una funzione che calcoli la probabilità condizionata
def probabilita_condizionata(bigram, tot_bigrams, tot_tokens):
  freq_bigramma = tot_bigrams.count(bigram)
  tot_elem_1 = tot_tokens.count(bigram[0])
  prob_condizionata = freq_bigramma/tot_elem_1
  return prob_condizionata

#Chiamo la funzione <liste> nella quale utilizzo la funzione <probabilita_condizionata> così da calcolarla su ogni elemento della lsta di tuple aggettivo/sostantivo
elem_prob_condizionata = liste(adj_noun_tuples, probabilita_condizionata, ngrams_2, file_tokens)

#Chiamo la funzione top_10 per trovare e stampare i primi 10 elementi per probabilità condizionata
top_10(elem_prob_condizionata, funzione = "probabilità condizionata")


#Adesso creo una funziona che calcoli la probabilità congiunta        
def probabilità_congiunta(bigram, tot_bigrams, tot_tokens):
  prob_cond = probabilita_condizionata(bigram, tot_bigrams, tot_tokens) #calcolo la probabilità condizionata usando la funzione precedente
  freq_elem_1 = tot_tokens.count(bigram[0])                             #calcolo la frequenza dell'elemento 1  
  prob_elem_1 = freq_elem_1/len(tot_tokens)                             #e la probabilità dell'elemento 1
  prob_congiunta = prob_cond * prob_elem_1                              #calcolo la probabilità congiunta
  return prob_congiunta

#Creo la lista di bigrammi con associata la funzione <probabilità_congiunta>
elem_prob_congiunta = liste(adj_noun_tuples, probabilità_congiunta, ngrams_2, file_tokens) #creiamo la lista di bigrammi con associata la prob congiunta

#Chiamo la funzione top_10
top_10(elem_prob_congiunta, funzione = "probabilità congiunta")


#Adesso creo una funziona che calcoli la mutual information
def MI(bigram, tot_bigrams, tot_tokens):
  freq_bigramma = tot_bigrams.count(bigram)
  
  freq_elem_1 = tot_tokens.count(bigram[0])
  freq_elem_2 = tot_tokens.count(bigram[1])
  
  MI = math.log((freq_bigramma*len(tot_tokens)/(freq_elem_1*freq_elem_2)), 2)   
  return MI

elem_MI = liste(adj_noun_tuples, MI, ngrams_2, file_tokens)              #creo la lista dei bigrammi con la loro MI
top10_MI = top_10(elem_MI, funzione = "mutual information") 

def LMI(bigram, tot_bigrams, tot_tokens):
  freq_bigramma = tot_bigrams.count(bigram)
  
  mutual_info = MI(bigram, tot_bigrams, tot_tokens)
  LMI = freq_bigramma * mutual_info    
  return LMI

  
elem_LMI = liste(adj_noun_tuples, LMI, ngrams_2, file_tokens)             #creo la lista dei bigrammi con relativa LMI
top10_LMI = top_10(elem_LMI, funzione = "local mutual information")       #ne stampo i primi dieci elementi


#La funzione sotto trova gli elementi in comune nelle liste di MI e LMI e li stampa, se ci sono
def elem_comuni_MI_LMI(MI_list, LMI_list):
  elem_comuni = []
  for elem in MI_list:
    for i in LMI_list:
      if elem[0] in i[0]:
        elem_comuni.append(elem)
  if len(elem_comuni) > 0:
    print(elem_comuni)
  else:
    print("Non ci sono elementi in comune")


elem_comuni_MI_LMI(top10_MI, top10_LMI)
        

###PUNTO CINQUE###
####MEDIA DELLE DISTRIBUZIONI DI FREQUENZA E MODELLO DI MARKOV

#Trovo le frasi con una lunghezza tra i dieci e venti caratteri
def frasi(sentences):
  sentences_10_20 = []
  for sentence in sentences:
    tokens = nltk.tokenize.word_tokenize(sentence)
    if 10<=len(tokens)<=20:
      sentences_10_20.append(sentence)
  return sentences_10_20

sentences_10_20 = frasi(file_sentences)

#Trovo gli hapax
def hapax(tokens):   
    hapax = []        
    vocab = list(set(tokens))       
    for token in vocab:        
        token_freq = tokens.count(token)
        if token_freq == 1:
          hapax.append(token) 
    return hapax
            
tot_hapax = hapax(file_tokens)

#Trovo le frasi nelle quali almeno la metà dei token compare due volte nel corpus
def frasi_few_hapax(sentences, tot_hapax):
  frasi = []
  for sentence in sentences:
    hapax_count = sum(sentence.count(token) for token in tot_hapax)
    if hapax_count < (len(sentence)/2):
      frasi.append(sentence)
  return frasi

frasi_no_hapax = frasi_few_hapax(sentences_10_20, tot_hapax)

#Faccio la media di distribuzione delle frequenze
def media_freq_token(sentences):
  tot_frequenze = []
  for sentence in sentences:
    tokens = nltk.tokenize.word_tokenize(sentence)
    freq_tot_token = FreqDist(tokens) 
    freq_media = sum(freq_tot_token.values())/len(tokens) 
    tot_frequenze.append((sentence, freq_media))
  return tot_frequenze

tot_frequenze = media_freq_token(frasi_no_hapax)

def media_max_min(list_of_tuples):
  frequenza_max = max(list_of_tuples, key = lambda x: x[1])
  frequenza_min = min(list_of_tuples, key = lambda x: x[1])
  print(f"Il valore massimo di frequenza media appartiene alla frase: {frequenza_max[0]}, con un valore di {frequenza_max[1]}")
  print(f"Il valore minimo di frequenza media appartiene alla frase: {frequenza_min[0]}, con un valore di {frequenza_min[1]}")
     
media_max_min(tot_frequenze)

#Modello markoviano del secondo ordine

#Trovo le frequenze di distribuzione dei token, trigrammie bigrammi
freq_tokens = nltk.FreqDist(file_tokens)
trigrams_freq_dist = nltk.FreqDist(ngrams_3)
bigrams_freq_dist = nltk.FreqDist(ngrams_2)

#La funzione sotto calcola la probabilità condizionata e sarà riusata nella funzione del modello di markov
def conditional_probability(trigram, distribuzione_freq_trigrammi, distribuzione_freq_bigrammi):
  if distribuzione_freq_bigrammi[(trigram[0], trigram[1])] == 0:
    conditional_prob = 0
  else:
    conditional_prob = distribuzione_freq_trigrammi[trigram]/distribuzione_freq_bigrammi[(trigram[0], trigram[1])]
  return conditional_prob

#La funzione sotto calcola la probabilità secondo un modello di markov di ordine 2
def markov_2(sentence, distribuzione_freq_trigrammi, distribuzione_freq_bigrammi, distribuzione_freq_tokens, corpus):
  tokens = nltk.tokenize.word_tokenize(sentence)
  trigrams_in_sentence = list(nltk.ngrams(tokens, 3))
    
  prob = 1.0*(distribuzione_freq_bigrammi[(tokens[0], tokens[1])]/distribuzione_freq_tokens[tokens[0]]) * (distribuzione_freq_tokens[tokens[0]]/corpus)
    
  for trigram in trigrams_in_sentence:
      cond_prob = conditional_probability(trigram, distribuzione_freq_trigrammi, distribuzione_freq_bigrammi)
      prob = prob*cond_prob
  return prob

#Calcolo la probabilità di ogni frase con il modello di markov
def f(sentences, distribuzione_freq_trigrammi, distribuzione_freq_bigrammi, distribuzione_freq_tokens):
  frasi_markov = []
  for sentence in sentences:
    prob = markov_2(sentence, distribuzione_freq_trigrammi, distribuzione_freq_bigrammi, distribuzione_freq_tokens, len(file_tokens))
    frasi_markov.append((sentence, prob))
  return frasi_markov

frasi_markov = f(sentences_10_20, trigrams_freq_dist, bigrams_freq_dist, freq_tokens)

#Trovo e stampo la frase con la probabilità maggiore e minore
def markov_max_min(list_of_tuples):
  prob_max = max(list_of_tuples, key = lambda x: x[1])
  prob_min = min(list_of_tuples, key = lambda x: x[1])
  print(f"La probabilità massima secondo il Modello di Markov di ordine due appartiene alla frase: {prob_max[0]}, con un valore di {prob_max[1]}")
  print(f"La probabilità minima secondo il Modello di Markov di ordine due appartiene alla frase: {prob_min[0]}, con un valore di {prob_min[1]}")
     
markov_max_min(frasi_markov)


###PUNTO SEI###
##ESTRAZIONE DELLE NAME ENTITIES E TOP 15 ELEMENTI PIU' FREQUENTI

#Trovo l'aòbero delle name entities a partire dalla lista delle part of speech del file
NE_tree = nltk.ne_chunk(file_POS)          

#Inizializzo un dizionario vuoto in cui mettere come chiavi le name entities e come valori le istanze di quella classe di NE
NE = {}      

#Per ogni nodo nell'albero di NE individuo le NE e le loro istanze, poi le inserisco nel dizionario se non ci sono di già                                   
for nodo in NE_tree: 
    if hasattr(nodo, 'label'): 
        entity_type = nodo.label() 
        entity = " ".join([token for token, POS in nodo.leaves()]) 
        if entity_type in NE.keys():           
            NE[entity_type].append(entity) 
        else:
            NE[entity_type] = [entity]  

#Creo delle liste che contengono solo i valori delle chiave del dizionario, ovvero le istanze delle classi di NE
person = NE['PERSON']                          
org = NE['ORGANIZATION']
GPE = NE['GPE']

#La funzione sotto calcola le frequenze delle istanze nelle classi create nello step precedente e stampa le 15 più frequenti
def top_15_NE(lista, entity_type, top_n = 15): 
   freq_NE = FreqDist(lista)
   for entity, freq in freq_NE.most_common(top_n):
      print(f"{entity_type}: {entity} \n\tFrequenza assoluta: {freq}")

top_15_NE(person, entity_type = 'PERSON', top_n = 15)
top_15_NE(org, entity_type = 'ORGANIZATION', top_n = 15)
top_15_NE(GPE, entity_type = 'GPE', top_n = 15)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


La lista token/lemmi del testo: [('The', 'The'), ('Project', 'Project'), ('Gutenberg', 'Gutenberg'), ('eBook', 'eBook'), ('of', 'of'), ('Mrs.', 'Mrs.'), ('Dalloway', 'Dalloway'), ('This', 'This'), ('ebook', 'ebook'), ('is', 'be'), ('for', 'for'), ('the', 'the'), ('use', 'use'), ('of', 'of'), ('anyone', 'anyone'), ('anywhere', 'anywhere'), ('in', 'in'), ('the', 'the'), ('United', 'United'), ('States', 'States'), ('and', 'and'), ('most', 'most'), ('other', 'other'), ('parts', 'part'), ('of', 'of'), ('the', 'the'), ('world', 'world'), ('at', 'at'), ('no', 'no'), ('cost', 'cost'), ('and', 'and'), ('with', 'with'), ('almost', 'almost'), ('no', 'no'), ('restrictions', 'restriction'), ('whatsoever', 'whatsoever'), ('.', '.'), ('You', 'You'), ('may', 'may'), ('copy', 'copy'), ('it', 'it'), (',', ','), ('give', 'give'), ('it', 'it'), ('away', 'away'), ('or', 'or'), ('re-use', 're-use'), ('it', 'it'), ('under', 'under'), ('the', 'the'), ('terms', 'term'), ('of', 'of'), ('the', 'the'), ('Project'